# What Rule-Based Detection Can and Cannot Do

agentdx is an open-source SDK that detects reasoning-level failure pathologies in
AI agent traces. It uses rule-based heuristics — pattern matching on trace structure,
not ML models or LLM judges.

This notebook is an honest walkthrough of what that approach catches and what it
misses. We ran 42 synthetic traces through all 7 detectors and present the results
with dual metrics (tolerant and strict) and 95% confidence intervals.

**This is a development validation, not an independent evaluation.** Traces were
generated to exhibit specific pathologies. Results validate that detectors fire on
intended patterns — nothing more.

We start with a miss, then a catch, then walk through all seven failure patterns.

## Setup

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on path
project_root = Path(".").resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from agentdx import Diagnoser, JSONParser, PATHOLOGY_REGISTRY, Pathology
from case_study.harness.evaluate import run_evaluation
from case_study.harness.metrics import (
    compute_per_detector_metrics,
    compute_aggregate_metrics,
    compute_cross_detector_interference,
    compute_strict_metrics,
)

parser = JSONParser()
diagnoser = Diagnoser()

TRACES_DIR = Path("traces")
print(f"agentdx loaded with {len(diagnoser.detectors)} detectors")
print(f"Traces directory: {TRACES_DIR}")

## The Miss: gh_03 — A Cooking Assistant Answers Quantum Physics

The system prompt says *"You are a cooking assistant."* The user asks about
carbonara, then quantum entanglement, then NVIDIA swing trading, then anglerfish
bioluminescence. The agent answers every question in expert detail.

This is an obvious goal hijack. Our detector **does not catch it.**

In [ ]:
trace = parser.parse(str(TRACES_DIR / "goal_hijacking/gh_03.json"))
report = diagnoser.diagnose(trace)

print(f"System prompt: {trace.system_prompt[:80]}...\n")

# Show user questions
user_msgs = [m for m in trace.messages if m.role.value == "user"]
for i, msg in enumerate(user_msgs):
    print(f"  Q{i + 1}: {msg.content[:80]}...")

gh_result = [r for r in report.results if r.pathology == Pathology.GOAL_HIJACKING][0]
print(f"\nDetected: {gh_result.detected}")
print(f"Confidence: {gh_result.confidence:.2f}")
print()
print("Why it missed: The user's messages contain no injection patterns —")
print("just polite questions in new domains. Detecting that 'quantum entanglement'")
print("is off-topic for a cooking assistant requires semantic similarity,")
print("not keyword matching.")

## The Catch: ht_03 — Real Stock Data Mixed with Fabricated Analyst Ratings

A financial analysis agent calls `stock_price` (succeeds) and `analyst_reports`
(fails with a service unavailability error). The agent then presents a comprehensive
analysis mixing real price data with entirely fabricated analyst consensus ratings,
price targets, and earnings estimates.

In [ ]:
trace = parser.parse(str(TRACES_DIR / "hallucinated_tool_success/ht_03.json"))
report = diagnoser.diagnose(trace)

# Show the tool calls
for msg in trace.messages:
    for tc in msg.tool_calls:
        status = "SUCCESS" if tc.success else f"FAILED: {tc.error_message}"
        print(f"  {tc.tool_name}({tc.arguments}) -> {status}")

ht_result = [r for r in report.results if r.pathology == Pathology.HALLUCINATED_TOOL_SUCCESS][0]
print(f"\nDetected: {ht_result.detected}")
print(f"Confidence: {ht_result.confidence:.2f}")
print(f"Severity: {ht_result.severity}")
print()
print("Why it caught this: The structural signal is mechanical — a failed tool")
print("call followed by an assistant message presenting specific data. Rule-based")
print("detection excels here because the pattern is structural, not semantic.")

## Seven Failure Patterns in Action

Below we demonstrate each of the 7 pathologies with a representative trace.
These are the patterns agentdx is designed to catch.

### 1. Tool Thrashing

An agent repeatedly calls the same tool with near-identical arguments,
failing to advance or try a meaningfully different approach.

**OWASP:** A04 — Unrestricted Tool Utilisation | **MAST:** Execution Failure

In [ ]:
trace = parser.parse(str(TRACES_DIR / "tool_thrashing/tt_01.json"))
report = diagnoser.diagnose(trace)

for msg in trace.messages:
    for tc in msg.tool_calls:
        print(f"  Step {msg.step_index}: {tc.tool_name}({tc.arguments}) -> {tc.result}")

tt_result = [r for r in report.results if r.pathology == Pathology.TOOL_THRASHING][0]
print(f"\nDetected: {tt_result.detected}")
print(f"Confidence: {tt_result.confidence:.2f}")
print(f"Severity: {tt_result.severity}")
print(f"Evidence: {tt_result.evidence}")

### 2. Context Erosion

The agent gradually loses reference to its system prompt and early
conversation context over the course of a long conversation.

**OWASP:** A05 — Improper Multi-Agent Orchestration | **MAST:** Task Derailment

In [ ]:
trace = parser.parse(str(TRACES_DIR / "context_erosion/ce_01.json"))
report = diagnoser.diagnose(trace)

print(f"System prompt: {trace.system_prompt[:120]}...\n")

assistant_msgs = [m for m in trace.messages if m.role.value == "assistant"]
print(f"Early response (turn 1): {assistant_msgs[0].content[:150]}...\n")
print(f"Late response (turn {len(assistant_msgs)}): {assistant_msgs[-1].content[:150]}...\n")

ce_result = [r for r in report.results if r.pathology == Pathology.CONTEXT_EROSION][0]
print(f"Detected: {ce_result.detected}")
print(f"Confidence: {ce_result.confidence:.2f}")
print(f"Severity: {ce_result.severity}")

### 3. Instruction Drift

The agent progressively deviates from its assigned role, with a
measurable decline in alignment with system prompt instructions over time.

**OWASP:** A02 — Privilege Compromise | **MAST:** Task Derailment

In [ ]:
trace = parser.parse(str(TRACES_DIR / "instruction_drift/id_01.json"))
report = diagnoser.diagnose(trace)

id_result = [r for r in report.results if r.pathology == Pathology.INSTRUCTION_DRIFT][0]
print(f"Detected: {id_result.detected}")
print(f"Confidence: {id_result.confidence:.2f}")
print(f"Severity: {id_result.severity}")
print(f"Description: {id_result.description}")

### 4. Recovery Blindness

The agent fails to acknowledge or act on tool errors, continuing
as if nothing went wrong.

**OWASP:** A08 — Inadequate Sandboxing | **MAST:** Execution Failure

In [ ]:
trace = parser.parse(str(TRACES_DIR / "recovery_blindness/rb_01.json"))
report = diagnoser.diagnose(trace)

for msg in trace.messages:
    for tc in msg.tool_calls:
        if not tc.success:
            print(f"Tool error: {tc.tool_name} -> {tc.error_message}")
assistant_msgs = [m for m in trace.messages if m.role.value == "assistant" and not m.tool_calls]
if assistant_msgs:
    print(f"Agent response: {assistant_msgs[0].content[:200]}...\n")

rb_result = [r for r in report.results if r.pathology == Pathology.RECOVERY_BLINDNESS][0]
print(f"Detected: {rb_result.detected}")
print(f"Confidence: {rb_result.confidence:.2f}")
print(f"Evidence: {rb_result.evidence}")

### 5. Hallucinated Tool Success

The agent claims a failed tool returned useful data, fabricating
results that were never received.

**OWASP:** A06 — Overreliance on Agentic | **MAST:** Execution Failure

In [ ]:
trace = parser.parse(str(TRACES_DIR / "hallucinated_tool_success/ht_01.json"))
report = diagnoser.diagnose(trace)

for msg in trace.messages:
    for tc in msg.tool_calls:
        if not tc.success:
            print(f"Tool: {tc.tool_name} -> FAILED: {tc.error_message}")
assistant_msgs = [m for m in trace.messages if m.role.value == "assistant" and not m.tool_calls]
for msg in assistant_msgs:
    if "according" in msg.content.lower() or "result" in msg.content.lower():
        print(f"\nAgent fabrication: {msg.content[:200]}...\n")
        break

ht_result = [r for r in report.results if r.pathology == Pathology.HALLUCINATED_TOOL_SUCCESS][0]
print(f"Detected: {ht_result.detected}")
print(f"Confidence: {ht_result.confidence:.2f}")
print(f"Severity: {ht_result.severity}")

### 6. Goal Hijacking

The agent's goal gets redirected by prompt injection or drastic
topic shifts.

**OWASP:** A01 — Agentic Prompt Injection | **MAST:** Task Derailment

In [ ]:
trace = parser.parse(str(TRACES_DIR / "goal_hijacking/gh_01.json"))
report = diagnoser.diagnose(trace)

gh_result = [r for r in report.results if r.pathology == Pathology.GOAL_HIJACKING][0]
print(f"Detected: {gh_result.detected}")
print(f"Confidence: {gh_result.confidence:.2f}")
print(f"Severity: {gh_result.severity}")
print(f"Evidence: {gh_result.evidence}")

### 7. Silent Degradation

Output quality declines progressively without explicit errors —
responses get shorter, more generic, and less specific over time.

**OWASP:** A10 — Misaligned Behaviours | **MAST:** Output Quality

In [ ]:
trace = parser.parse(str(TRACES_DIR / "silent_degradation/sd_01.json"))
report = diagnoser.diagnose(trace)

assistant_msgs = [m for m in trace.messages if m.role.value == "assistant"]
print(f"Early response length: {len(assistant_msgs[0].content)} chars")
print(f"Late response length:  {len(assistant_msgs[-1].content)} chars\n")

sd_result = [r for r in report.results if r.pathology == Pathology.SILENT_DEGRADATION][0]
print(f"Detected: {sd_result.detected}")
print(f"Confidence: {sd_result.confidence:.2f}")
print(f"Description: {sd_result.description}")

## Aggregate Results — Dual Metrics with Confidence Intervals

We present two metric views:
- **Tolerant**: expected cross-detector co-occurrences excluded from FP count
- **Strict**: every unexpected detection counted as FP

With 3-6 traces per detector, confidence intervals are wide. A recall of
1.00 with N=3 has a 95% CI of [0.29, 1.00].

In [ ]:
results = run_evaluation(TRACES_DIR, TRACES_DIR / "ground_truth.json")
per_detector = compute_per_detector_metrics(results)
aggregate = compute_aggregate_metrics(per_detector)
strict_det, strict_agg = compute_strict_metrics(per_detector)

# Dual aggregate table
print("=" * 60)
print(f"{'Metric':<25} {'Tolerant':>10} {'Strict':>10}")
print("=" * 60)
for label, key in [
    ("Macro Precision", "macro_precision"),
    ("Macro Recall", "macro_recall"),
    ("Macro F1", "macro_f1"),
    ("Micro Precision", "micro_precision"),
    ("Micro Recall", "micro_recall"),
    ("Micro F1", "micro_f1"),
]:
    print(f"{label:<25} {aggregate[key]:>10.3f} {strict_agg[key]:>10.3f}")
print("=" * 60)

In [ ]:
# Per-detector tolerant view with CIs
print("=" * 100)
print(
    f"{'Detector':<30} {'P':>6} {'R':>6} {'R 95% CI':>16} {'F1':>6} {'TP':>4} {'FP':>4} {'TN':>4} {'FN':>4} {'Tol':>4}"
)
print("=" * 100)
for key, m in per_detector.items():
    info = PATHOLOGY_REGISTRY.get(Pathology(key))
    name = info.name if info else key
    r_lo, r_hi = m.recall_ci
    ci_str = f"[{r_lo:.2f}, {r_hi:.2f}]" if (m.tp + m.fn) > 0 else "      —"
    print(
        f"{name:<30} {m.precision:>6.2f} {m.recall:>6.2f} {ci_str:>16} {m.f1:>6.2f} "
        f"{m.tp:>4} {m.fp:>4} {m.tn:>4} {m.fn:>4} {m.tolerated:>4}"
    )
print("=" * 100)

print()
print("Strict view (tolerated -> FP):")
print("-" * 80)
print(f"{'Detector':<30} {'P':>6} {'R':>6} {'F1':>6} {'TP':>4} {'FP':>4} {'TN':>4} {'FN':>4}")
print("-" * 80)
for key, m in strict_det.items():
    info = PATHOLOGY_REGISTRY.get(Pathology(key))
    name = info.name if info else key
    print(
        f"{name:<30} {m.precision:>6.2f} {m.recall:>6.2f} {m.f1:>6.2f} "
        f"{m.tp:>4} {m.fp:>4} {m.tn:>4} {m.fn:>4}"
    )

## Cross-Detector Interference — A Finding, Not a Bug

Agent failures don't occur in isolation — they cluster. When one pathology is
present, related detectors often fire. Rather than hiding this behind tolerations,
we present it as a finding about how agent failures co-occur in practice.

The interference matrix shows, for each row (the target pathology a trace was
designed for), how many times each column detector fired.

In [ ]:
interference = compute_cross_detector_interference(results)

short = {
    "tool_thrashing": "TT",
    "context_erosion": "CE",
    "instruction_drift": "ID",
    "recovery_blindness": "RB",
    "hallucinated_tool_success": "HT",
    "goal_hijacking": "GH",
    "silent_degradation": "SD",
}
keys = list(short.keys())

header = f"{'Target':<6} | " + " | ".join(f"{short[k]:>4}" for k in keys)
sep = "-" * len(header)
print(header)
print(sep)
for row in keys:
    cells_str = " | ".join(f"{interference.get(row, {}).get(col, 0):>4}" for col in keys)
    print(f"{short[row]:<6} | {cells_str}")

print()
total_tolerated = sum(m.tolerated for m in per_detector.values())
print(f"Total tolerated cross-fire detections: {total_tolerated}")
print()
print("Key co-occurrence patterns:")
print("  RB/HTS: Both fire on failed tools — RB checks if error is ignored,")
print("          HTS checks if results are fabricated.")
print("  ID/CE:  Drifting from role naturally means losing system prompt vocabulary.")
print("  TT/RB:  Retry loops often fail to recover from the underlying error.")

## Limitations — What This Evaluation Does Not Show

### What we validated
- Detectors fire on traces designed to exhibit their target pathology
- Detectors generally don't fire on traces designed for other pathologies
- Cross-detector co-occurrence patterns match expected co-fire behaviour

### What we did not validate
- **No production traces.** All 42 traces are synthetic, induced by LLM agents.
  Real-world failures may differ.
- **No held-out data.** All traces were visible during development.
- **No baselines.** We haven't compared against random detectors or keyword heuristics.
- **No adversarial testing.** We haven't tested traces designed to evade detection.
- **Wide confidence intervals.** With 3-6 traces per detector, recall CI of [0.29, 1.00]
  for 3/3 detections is representative.
- **Tolerated detections are post-hoc.** The tolerance list was written after observing
  cross-fire, by the same team that wrote the detectors.
- **Circular validation.** Generating agents received behavioural descriptions
  functionally equivalent to detector specifications.

### The honest claim

agentdx provides a rule-based diagnostic toolkit for identifying 7 common AI agent
failure patterns. In development testing, the detectors correctly identified intended
patterns with high precision. The toolkit is best used as a **debugging aid during
development**, not as a production monitoring system.

---

*Generated with [agentdx](https://pypi.org/project/agentdx/) v0.1.0a2*